In [1]:
import napari
import numpy as np
from aicsimageio import AICSImage

import pandas as pd
from skimage import measure, io

import tifffile as tf                # not 'import tifffile as from ...'
from tifffile import TiffFile        # if you need the TiffFile class

import xarray as xr
import dask.array as da              # instead of 'import dask.arraytf'

import matplotlib.pyplot as plt

import tifftools

import os
from os.path import sep

from PIL import Image
import imageio

from skimage.measure import regionprops_table

import seaborn as sns
import czifile

import math

/Users/fisherguest/miniconda3/envs/sansachen_czi/lib/python3.11/site-packages/pydantic/_migration.py:283: UserWarning: `pydantic.error_wrappers:ValidationError` has been moved to `pydantic:ValidationError`.
  warnings.warn(f'`{import_path}` has been moved to `{new_location}`.')


In [5]:
# load the files
fly3 = '/Users/fisherguest/Downloads/sansa images/03222025/SC-0322-fly3.czi'


In [6]:
# import image:
img = AICSImage(fly3)
# Get the data in (C, Z, Y, X)
# C: Channels (e.g., the different lasers/fluorophores); Z: Z-stacks (slices in depth); Y and X: The spatial dimensions (height and width)
# i don't have S or T.
data = img.get_image_data("CZYX", S=0, T=0)
# checked using "previous TQ method": 568 is first channel, 488 is second channel.
red_channel   = data[0]
green_channel = data[1]

In [7]:
# open image with set contrast limits (so don't darg the bar):
viewer = napari.Viewer()
viewer.add_image(
    red_channel,
    name='568 Channel',
    colormap='red',
    blending='additive',
    contrast_limits=(100, 2500)  # if right-click on the contrast-limits bar, will see this upper and lower boundary; if not set, will be the original values.
) # so to find the best limits range, open the original image first to see the
viewer.add_image(
    green_channel,
    name='488 Channel',
    colormap='green',
    blending='additive',
    contrast_limits=(50, 800)   # specify a different range for the green channel
)
napari.run()

ImportError: No Qt bindings could be found.

napari requires either PyQt5 (default) or PySide2 to be installed in the environment.

With pip, you can install either with:
  $ pip install -U 'napari[all]'  # default choice
  $ pip install -U 'napari[pyqt5]'
  $ pip install -U 'napari[pyside2]'

With conda, you need to do:
  $ conda install -c conda-forge pyqt
  $ conda install -c conda-forge pyside2

Our heuristics suggest you are using 'conda' to manage your packages.

In [ ]:
# File paths
label_file = '/Users/fisherguest/Downloads/sansa images/03222025/roi/fly3-new-new.tif'
# Load the label mask (the TIFF file you saved from napari)
label_mask = tf.imread(label_file)
# for the label tif file obtained in previous TQ method, need the below precessing:
#squeezed_mask = np.squeeze(label_mask) # so change from shape: (1, 2, Z, Y, X) to (2, Z, Y, X)
#squeezed_mask_green = squeezed_mask[1] # so only extract the green (second) channel.
print("Label mask shape:", label_mask.shape)
print("Green channel shape:", green_channel.shape)
# Check that the dimensions match (e.g., both are 3D)
if green_channel.shape != label_mask.shape:
    raise ValueError("The dimensions of the green channel and the label mask do not match. "
                     "Please verify that your label mask corresponds to the correct z, y, and x dimensions.")

In [ ]:
# check which z stack I drew the label on.
sums = label_mask.sum(axis=(1,2))
painted_slices = np.where(sums > 0)[0]
print("You painted on slice(s):", painted_slices)

In [ ]:
z_ranges = [
    (16, 63),  # glomerulus 1: use slices 10 through 20
    (56, 94),  #2
    (12, 22),  #3
    (12, 22),  #4
    (12, 22),  #5
    (12, 22),  #6
    (12, 22),  #7
    (12, 22),  #8
    (12, 22),  #9
    (12, 22),  #10
    (12, 22),  #11
    (12, 22),  #12
    (12, 22),  #13
    (12, 22),  #14
    (12, 22),  #15
    (12, 22),  #16
    (12, 22),  #17
    (30, 45),  #18
]

In [ ]:
z_ranges = [
    (16, 63),  # glomerulus 1: use slices 10 through 20
    (56, 94)]  #2

In [4]:
red_threshold_lower = 250

In [ ]:
n_glom    = int(label_mask.max())  # should be 18
mean_red   = np.zeros(n_glom, dtype=float)
mean_green = np.zeros(n_glom, dtype=float)

for i in range(1, n_glom+1):
    # 1) find the slice you painted on:
    zs, ys, xs = np.where(label_mask == i)
    z_draw = zs[0]
    
    # 2) extract the 2D ROI on that slice:
    roi2d = (label_mask[z_draw] == i)   # (Y, X), bool
    
    # 3) build a 1D mask for your Z‑range:
    z0, z1   = z_ranges[i-1]
    Z, Y, X  = label_mask.shape
    mask_z   = (np.arange(Z) >= z0) & (np.arange(Z) <= z1)  # (Z,)
    
    # 4) broadcast to 3D:
    roi = mask_z[:, None, None] & roi2d[None, :, :]         # (Z, Y, X), bool
    # — b) zero out outside your propagated ROI —
    red_sub   = red_channel[z0:z1+1]   * roi[z0:z1+1]
    green_sub = green_channel[z0:z1+1]
    # — c) threshold & extract means —
    mask      = (red_sub > red_threshold_lower)
    red_vals   = red_sub  [mask]
    green_vals = green_sub[mask]
    mean_red[i-1]   = red_vals.mean()   if red_vals.size   else np.nan
    mean_green[i-1] = green_vals.mean() if green_vals.size else np.nan
